# Romeo empático con Soup (Colab)

Entrena `Qwen2.5-0.5B-Instruct` con LoRA usando **Soup** y lo exporta a GGUF para Ollama.

En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*. Sube `soup.yaml`, `Modelfile` y `data/romeo_empatico.jsonl` (o el zip entero y descomprímelo en la celda 2).

In [ ]:
%%capture
!pip install "soup-cli[train]"

In [ ]:
import os
if not os.path.exists("soup.yaml"):
    from google.colab import files
    subido = files.upload()          # sube romeo_soup.zip
    !unzip -o -q romeo_soup.zip
!soup doctor
!soup train --config soup.yaml --dry-run

## Entrenar (~10 min en una T4)

In [ ]:
!soup train --config soup.yaml

## Probar unas frases

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
SISTEMA = "Eres Romeo, un robot amigable que conversa en voz alta con niños y jóvenes. Responde en español, tuteando, con una o dos frases cortas, con calidez y empatía."
base = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
m = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float16).to("cuda"),
                              "output/romeo-empatico")
for t in ["Hoy me siento muy solo, nadie quiso jugar conmigo en el recreo.",
          "¡Saqué un diez en mi examen de matemáticas!",
          "Mi perrito se murió ayer.",
          "Me da miedo la oscuridad cuando me voy a dormir.",
          "Estoy enojado porque mi hermano rompió mi juguete."]:
    ids = tok.apply_chat_template([{"role": "system", "content": SISTEMA}, {"role": "user", "content": t}],
                                  add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = m.generate(ids, max_new_tokens=60, do_sample=True, temperature=0.3)
    print(f"Usuario: {t}\nRomeo:   {tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()}\n")

## Exportar a GGUF y descargar

In [ ]:
!soup export --model output/romeo-empatico --format gguf --quant q8_0 -o romeo-empatico.q8_0.gguf
!zip -q romeo_ollama.zip romeo-empatico.q8_0.gguf Modelfile && ls -lh romeo_ollama.zip
from google.colab import files
files.download("romeo_ollama.zip")

## En la PC de Ollama

```
ollama create romeo-empatico -f Modelfile
ollama run romeo-empatico "Hoy me siento muy solo"
```

Y arranca el robot con `chat_model:=romeo-empatico`.